# Gaiatools


gaiatools is a Python library that simplifies access to the ESA Gaia DR3 stellar catalog by building on top of astroquery. It allows users to retrieve stars from any region of the sky, filter them by brightness and distance, generate interactive Hertzsprung-Russell diagrams and 3D sky maps, detect stellar clusters using HDBSCAN, and cross-match with the NASA Exoplanet Archive to identify stars with confirmed exoplanets.





In [ ]:
!pip install gaiatools astroquery

In [ ]:
!pip install gaiatools==0.1.1


# 1.Why gaiatools?
gaiatools allows you to query any region of the sky by specifying the center coordinates (RA, Dec) and a search radius in degrees.



Without gaiatools

In [ ]:
from astroquery.gaia import Gaia

query = """SELECT TOP 2000 source_id, ra, dec, phot_g_mean_mag, phot_bp_mean_mag,
phot_rp_mean_mag, parallax, pmra, pmdec
FROM gaiadr3.gaia_source
WHERE CONTAINS(
   POINT('ICRS', ra, dec),
    CIRCLE('ICRS', 56.75, 24.12, 2.0)
) = 1"""

job = Gaia.launch_job(query)
results = job.get_results().to_pandas()

The archive is unstable and may perform below expectations. If launching multiple, consecutive, heavy queries through Python, please space them out (e.g., using sleep(1)) to avoid overloading the system. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the December 2025 infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


With gaiatools

In [ ]:
from gaiatools import StarCatalog

catalog = StarCatalog.from_region(ra=56.75, dec=24.12, radius=2.0, limit=10000)
print(catalog)

The catalog has 10000 stars


# 2.Filtering stars

In [ ]:
# Filter by brightness - only stars brighter than magnitude 18
bright_stars = catalog.filter_by_band("phot_g_mean_mag", max_mag=18)
print(f"Stars brighter than mag 18: {len(bright_stars)}")

# Filter by distance - only stars closer than 500 pc (parallax > 2 mas)
nearby_stars = catalog.filter_by_distance(min_parallax=2)
print(f"Stars closer than 500 pc: {len(nearby_stars)}")

Stars brighter than mag 18: 563
Stars closer than 500 pc: 171


# 3.H-R Diagram
The H-R diagram plots stars by their color (temperature) vs absolute magnitude (luminosity).
Stars follow distinct patterns: the main sequence, red giants, and white dwarfs.*texto en cursiva*

In [ ]:
# Plot interactive H-R diagram with absolute magnitude
catalog.plot_hr()

### 3.1 Stellar cluster detection

In [ ]:
# Detect clusters
clusters = catalog.hdbscan()

# Plot clusters
catalog.plot_clusters(clusters)

/usr/local/lib/python3.12/dist-packages/gaiatools/plot.py:64: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



# 4. Cross-matching with nasa exoplanet archive
gaiatools can cross-match your stellar catalog with the NASA Exoplanet Archive
to identify which stars in your region have confirmed exoplanets.

In [ ]:
from gaiatools.exoplanets import get_exoplanets, crossmatch, plot_system

# Get all confirmed exoplanets from NASA
exoplanets = get_exoplanets()

# Query the Kepler field - rich in exoplanet host stars
kepler_catalog = StarCatalog.from_region(ra=291.0, dec=44.5, radius=2.0, limit=2000)

# Cross-match with NASA Exoplanet Archive
matches = crossmatch(kepler_catalog.data, exoplanets)

if len(matches) > 0:
    print(matches[["ra", "dec", "pl_name"]])
else:
    print("No exoplanet host stars found in this region.")

Total estrellas Gaia: 2000
Total exoplanetas NASA: 4590
Distancia mínima encontrada: 0.006109599628434091 arcsec
Matches < 1 arcsec: 2
              ra        dec        pl_name
743   293.299926  43.555277  Kepler-1868 b
1114  293.288023  43.514904  Kepler-1579 b
1701  293.287286  43.513944  Kepler-1579 b
1904  293.298374  43.555801  Kepler-1868 b


In [ ]:
# Show exoplanet details for matched stars
if len(matches) > 0:
    star_names = matches["pl_name"].str.split(" ").str[:-1].str.join(" ").unique()
    details = exoplanets[exoplanets["hostname"].isin(star_names)]
    print(details[["hostname", "pl_name", "pl_orbsmax", "pl_masse", "pl_rade", "pl_orbper"]])

         hostname        pl_name  pl_orbsmax  pl_masse   pl_rade   pl_orbper
3584  Kepler-1579  Kepler-1579 b     0.01400       NaN  0.840000    0.849908
4529  Kepler-1868  Kepler-1868 b     0.62348       NaN  3.149501  211.033997


In [ ]:
plot_system("55 Cnc", exoplanets)

# 5. Get Sky Map
gaiatools can visualize your stars in a 3D interactive map using right ascension,
declination and distance. Stars are colored by their BP-RP color index.

In [ ]:
catalog.plot_sky()

## 6. Extracting a Specific Cluster

Once you've detected clusters, you can extract a specific one and work with it independently.

In [ ]:
# Extract cluster 0
cluster_0 = catalog.get_cluster(clusters, 1)
print(cluster_0)

# Plot its H-R diagram
cluster_0.plot_hr()

The catalog has 99 stars
